# ML utils

> Some helper functions for machine learning tasks.

In [ ]:
#| default_exp agents.ml_utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

from typing import  List, Tuple, Literal
import torch

from typing import Callable, List

import numpy as np
from torch import nn as nn
from torch.nn.utils import weight_norm

In [ ]:
#| export

class LRSchedulerPerStep():
    """
    Learning rate scheduler from Attention is all you need paper (https://arxiv.org/abs/1706.03762)
    One ajustment: Added base LR as tunable parameter rather than setting it automated based on model dimension
    """
    
    def __init__(self,
                optimizer: torch.optim.Optimizer, # Optimizer to adjust learning rate for
                base_learning_rate: float = 0.0001,
                warmup: int =4000):

        # Ensure optimizer is a PyTorch optimizer
        if not isinstance(optimizer, torch.optim.Optimizer):
            raise ValueError('Optimizer must be a PyTorch optimizer')
        
        self.optimizer = optimizer
        self.basic = base_learning_rate
        self.warm = warmup**-1.5
        self.scaling_factor = 1/warmup**-0.5 # ensures that the peak realtive to the base lr is always 1

        self.step_num = 0   
        self.step()
        
    def step(self):
        self.step_num += 1
        lr = self.basic * self.scaling_factor * min(self.step_num**-0.5, self.step_num*self.warm)
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

In [ ]:
#| export
def init_gru(input_size: int, recurrent_state_size: int) -> nn.Module:
    """
    Initialize a GRU module.

    Args:
        input_size (int): Input size to the GRU.
        recurrent_state_size (int): Recurrent state size for the GRU.

    Returns:
        nn.Module
    """
    gru = nn.GRU(input_size, recurrent_state_size)

    for name, param in gru.named_parameters():
        if "bias" in name:
            nn.init.constant_(param, 0)
        elif "weight" in name:
            nn.init.orthogonal_(param)

    return gru

def init_module(
    module: nn.Module, weight_init: Callable, bias_init: Callable, gain: float = 1.0
) -> nn.Module:
    """
    Initialize a module with the given weight and bias functions.

    Args:
        module (nn.Module): Module that is to be initialized with the given weight and bias.
        weight_init (Callable): Function for initializing weights.
        bias_init (Callable): Function for initialize biases.
        gain (float): Gain amount.

    Returns:
        nn.Module
    """
    weight_init(module.weight.data, gain=gain)
    bias_init(module.bias.data)
    weight_norm(module)

    return module


def init_mlp(input_size: int, hidden_sizes: List[int]) -> nn.Sequential:
    """
    Initialize the value head for the critic.

    Args:
        input_size (List[int]): Size of the recurrent state in the base RNN.
        hidden_sizes (List[int]): Sizes of the hidden layers of the MLP.

    Returns:
        nn.Sequential
    """

    def _init_orthogonal(m: nn.Module):
        return init_module(
            m, nn.init.orthogonal_, lambda x: nn.init.constant_(x, 0), np.sqrt(2)
        )

    feature_sizes = list([input_size])
    feature_sizes.extend(hidden_sizes)

    mlp_modules = list()
    for i in range(len(feature_sizes) - 1):
        hidden_layer = _init_orthogonal(
            nn.Linear(feature_sizes[i], feature_sizes[i + 1])
        )

        # zero bias
        torch.nn.init.zeros_(hidden_layer.bias)
        mlp_modules.append(hidden_layer)

        # relu
        mlp_modules.append(nn.ReLU())
        pass

    return nn.Sequential(*mlp_modules)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()